# v02 — Regularização

Mudança única em relação à v01: ajuste do `C` da regressão logística (mesmas 17 variáveis, mesmos folds).

Com poucas partidas por temporada e variáveis correlacionadas (Elo, forma, pontos), a hipótese é que regularizar mais reduz o excesso de confiança.

In [1]:
import warnings
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)

DATA = Path('../data')
SUB_DIR = Path('../submissions')
SUB_DIR.mkdir(exist_ok=True)
LOG_PATH = Path('../experimentos.csv')

CLASS_ORDER = ['A', 'D', 'H']
RANDOM_STATE = 42
FOLDS = [2016, 2017, 2018, 2019, 2020, 2021]  # walk-forward: treino = temporadas anteriores

train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
print(train.shape, test.shape)

(3419, 26) (1725, 22)


In [2]:
NUM_FEATURES = ['elo_home', 'elo_away', 'elo_diff', 'form_pts_home', 'form_pts_away',
                'form_gf_home', 'form_ga_home', 'form_gf_away', 'form_ga_away',
                'home_form_as_host', 'away_form_as_visitor', 'rest_days_home', 'rest_days_away',
                'season_ppg_home', 'season_ppg_away', 'round_n', 'h2h_home_pts']


def add_features(df):
    """Features derivadas usando apenas colunas da propria linha."""
    out = df.copy()
    out['venue_form_diff'] = out['home_form_as_host'] - out['away_form_as_visitor']
    out['form_pts_diff'] = out['form_pts_home'] - out['form_pts_away']
    out['gd_diff'] = (out['form_gf_home'] - out['form_ga_home']) - (out['form_gf_away'] - out['form_ga_away'])
    return out


train = add_features(train)
test = add_features(test)

In [3]:
def make_logreg(num_cols, cat_cols=(), C=1.0):
    transformers = [('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                                      ('scale', StandardScaler())]), list(num_cols))]
    if cat_cols:
        transformers.append(('cat', OneHotEncoder(handle_unknown='ignore'), list(cat_cols)))
    return Pipeline([('prep', ColumnTransformer(transformers)),
                     ('model', LogisticRegression(C=C, max_iter=5000, random_state=RANDOM_STATE))])


def predict_ordered(model, X):
    """Alinha predict_proba a ['A', 'D', 'H'] pelas classes do modelo."""
    raw = pd.DataFrame(model.predict_proba(X), columns=model.named_steps['model'].classes_)
    return raw.reindex(columns=CLASS_ORDER, fill_value=0.0).values


def walk_forward(build, cols):
    """Previsoes out-of-fold por temporada; o modelo e reconstruido do zero a cada fold."""
    preds, ys = {}, {}
    for season in FOLDS:
        tr = train[train['Season'] < season]
        va = train[train['Season'] == season]
        model = build()
        model.fit(tr[cols], tr['Res'])
        preds[season] = predict_ordered(model, va[cols])
        ys[season] = va['Res'].values
    return preds, ys


def summarize(preds, ys, name):
    rows = [{'Season': s, 'n': len(ys[s]), 'LogLoss': log_loss(ys[s], preds[s], labels=CLASS_ORDER)}
            for s in FOLDS]
    df = pd.DataFrame(rows)
    mean = np.average(df['LogLoss'], weights=df['n'])
    print(f'{name}: media ponderada = {mean:.4f} | desvio entre folds = {df.LogLoss.std():.4f}')
    return df, mean

In [4]:
def save_submission(proba, path):
    sub = pd.DataFrame(proba, columns=CLASS_ORDER)
    sub.insert(0, 'Id', test['Id'].values)
    assert list(sub.columns) == ['Id', 'A', 'D', 'H']
    assert len(sub) == len(test) and (sub['Id'].values == test['Id'].values).all()
    vals = sub[CLASS_ORDER].values
    assert np.isfinite(vals).all() and (vals > 0).all()
    assert np.allclose(vals.sum(axis=1), 1.0, atol=1e-9)
    sub.to_csv(path, index=False)
    print('salvo em', path, sub.shape)
    return sub


def log_experiment(version, change, features, model_desc, params, folds_df, mean, decision, csv_path):
    row = {'versao': version, 'data': date.today().isoformat(), 'mudanca': change,
           'features': features, 'modelo': model_desc, 'parametros': params,
           'folds': ','.join(map(str, FOLDS)),
           'logloss_por_fold': ';'.join(f'{s}:{v:.4f}' for s, v in zip(folds_df.Season, folds_df.LogLoss)),
           'logloss_medio': round(mean, 4), 'desvio': round(folds_df.LogLoss.std(), 4),
           'decisao': decision, 'csv': csv_path}
    log = pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame(columns=list(row.keys()))
    log = log[log['versao'] != version]
    log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
    log.to_csv(LOG_PATH, index=False)
    return log

In [5]:
results = {}
for C in [1.0, 0.3, 0.1, 0.03, 0.01, 0.003, 0.001]:
    preds, ys = walk_forward(lambda: make_logreg(NUM_FEATURES, C=C), NUM_FEATURES)
    results[C] = summarize(preds, ys, f'C={C}')

C=1.0: media ponderada = 1.0337 | desvio entre folds = 0.0455
C=0.3: media ponderada = 1.0334 | desvio entre folds = 0.0454


C=0.1: media ponderada = 1.0328 | desvio entre folds = 0.0452
C=0.03: media ponderada = 1.0312 | desvio entre folds = 0.0444


C=0.01: media ponderada = 1.0291 | desvio entre folds = 0.0429
C=0.003: media ponderada = 1.0273 | desvio entre folds = 0.0403


C=0.001: media ponderada = 1.0282 | desvio entre folds = 0.0372


In [6]:
best_C = min(results, key=lambda c: results[c][1])
best_df, best_mean = results[best_C]
print('melhor C:', best_C, '| v01 (C=1):', round(results[1.0][1], 4), '| v02:', round(best_mean, 4))
pd.DataFrame({c: r[0].LogLoss.values for c, r in results.items()}, index=FOLDS).round(4)

melhor C: 0.003 | v01 (C=1): 1.0337 | v02: 1.0273


,1.000,0.300,0.100,0.030,0.010,0.003,0.001
2016,1.0119,1.0113,1.0098,1.0067,1.0035,1.0017,1.0027
2017,1.1087,1.1083,1.1072,1.1040,1.0989,1.0924,1.0873
2018,0.9857,0.9855,0.9852,0.9844,0.9836,0.9839,0.9874
2019,0.9977,0.9977,0.9978,0.9981,0.9987,1.0008,1.0063
2020,1.0588,1.0586,1.0580,1.0566,1.0542,1.0521,1.0526
2021,1.0395,1.0393,1.0388,1.0375,1.0353,1.0328,1.0331


O LogLoss cai conforme `C` diminui até o ponto ótimo e depois volta a subir. A melhora aparece na maioria dos folds, então a v02 substitui a v01.

In [7]:
final = make_logreg(NUM_FEATURES, C=best_C).fit(train[NUM_FEATURES], train['Res'])
sub = save_submission(predict_ordered(final, test[NUM_FEATURES]), SUB_DIR / 'submission_v02.csv')
log_experiment('v02', 'ajuste de C', '17 numericas', 'LogisticRegression', f'C={best_C}',
               best_df, best_mean, 'promovido', 'submissions/submission_v02.csv')

salvo em ..\submissions\submission_v02.csv (1725, 4)


,versao,data,mudanca,features,modelo,parametros,folds,logloss_por_fold,logloss_medio,desvio,decisao,csv
0,v01,2026-09-28,baseline logreg,17 numericas,LogisticRegression,C=1,"2016,2017,2018,2019,2020,2021",2016:1.0119;2017:1.1087;2018:0.9857;2019:0.997...,1.0337,0.0455,campeao inicial,submissions/submission_v01.csv
1,v02,2026-09-28,ajuste de C,17 numericas,LogisticRegression,C=0.003,"2016,2017,2018,2019,2020,2021",2016:1.0017;2017:1.0924;2018:0.9839;2019:1.000...,1.0273,0.0403,promovido,submissions/submission_v02.csv
